<a href="https://colab.research.google.com/github/arauch6363-crypto/pt/blob/main/PT_github_2026_today_update.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install selenium
!pip install unidecode
!pip install fastparquet
!pip install google-colab-selenium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.0/512.0 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 8.1 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 21.6 MB/s eta 0:00:00


In [ ]:
#set mode

mode = 'get_todays_races'
#mode = 'get_past_results'
#mode = 'get_both'

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# Cell 3 - Imports
from datetime import date, datetime, timedelta
import pytz
import json
import re
import pandas as pd
from unidecode import unidecode
import numpy as np
import sys
import fastparquet

ModuleNotFoundError: No module named 'unidecode'

In [ ]:

# Cell 4 - Helper functions
import asyncio
import nest_asyncio
from playwright.async_api import async_playwright
nest_asyncio.apply()

async def get_web_content_async(url, retries=3):
    for attempt in range(retries):
        try:
            async with async_playwright() as p:
                browser = await p.chromium.launch(headless=True)
                page = await browser.new_page()
                try:
                    await page.goto(url, wait_until='commit', timeout=60000)
                    await page.wait_for_selector('#__NEXT_DATA__', state='attached', timeout=60000)
                    content = await page.evaluate('document.getElementById("__NEXT_DATA__").textContent')
                    return json.loads(content)
                finally:
                    await browser.close()
        except Exception as e:
            print(f"Attempt {attempt + 1} failed for {url}: {e}")
            if attempt == retries - 1:
                raise
            await asyncio.sleep(3)

def get_web_content(url, driver=None):
    loop = asyncio.get_event_loop()
    return loop.run_until_complete(get_web_content_async(url))

def web_driver():
    return None

def get_operator_data_odds(data, operators_priority=['PMU', 'PMU.fr', 'genybet']):
    for operator in operators_priority:
        for race in data:
            if race.get('operator') == operator:
                return race.get('runners', {})
    return None

def get_operator_data_dividends(data, operators_priority=['PMU', 'PMU.fr', 'genybet']):
    for operator in operators_priority:
        for race in data:
            if race.get('operator') == operator:
                return race.get('betDividends', {})
    return None

def generate_top_5_table(df, group_column):
    df_wins = df[df['ranking'] == 1]
    total_runs = df.groupby(group_column).size()
    win_count = df_wins.groupby(group_column).size()
    result = pd.DataFrame({
        'wins': win_count,
        'runs': total_runs
    }).fillna(0)
    result['win_percentage'] = (result['wins'] / result['runs']) * 100
    top_5 = result.sort_values(by='wins', ascending=False).head(20)
    top_5['formatted'] = top_5.apply(
        lambda row: f"{row.name} {int(row['runs'])}/{int(row['wins'])} {row['win_percentage']:.0f}%", axis=1
    )
    return top_5['formatted']

In [ ]:
def get_today_update():

    driver = None  # not needed with Playwright

    try:
        loadDate = datetime.today().strftime('%Y-%m-%d')
        my_tz = pytz.timezone('Europe/Berlin')
        now = datetime.now(my_tz)

        print('loadDate:', loadDate)
        print("Running today's update...")

        # Load all tables
        race_df_tdy = pd.read_parquet("./races_tdy.parquet")
        runners_tdy = pd.read_parquet("./runners_tdy.parquet")
        webTips_tdy = pd.read_parquet("./webTips_tdy.parquet")
        odds_tdy = pd.read_parquet("./odds_tdy.parquet")

        # Filter races within 5 hours but not within 45 minutes
        race_df_tdy['start_datetime'] = pd.to_datetime(
            race_df_tdy['date'].astype(str) + ' ' + race_df_tdy['time'].astype(str)
        )
        race_df_tdy['start_datetime'] = race_df_tdy['start_datetime'].dt.tz_localize('Europe/Berlin')

        races_to_update = race_df_tdy[
            (race_df_tdy['start_datetime'] > now + timedelta(minutes=45)) &
            (race_df_tdy['start_datetime'] <= now + timedelta(hours=5))
        ]

        print(f"Races to update: {len(races_to_update)}")

        if races_to_update.empty:
            print("No races in update window, exiting.")
            return False

        new_runners_dfs = []
        new_webTips_dfs = []
        new_odds_dfs = []

        for i, row in races_to_update.iterrows():
            url = row['race_url']
            race_id = str(row['id_race'])
            meeting_id = row['meetingId']

            print(f"Updating: {url}")

            props = get_web_content(url)

            # Runners
            try:
                runners = props['props']['pageProps']['initialState']['raceCardsState']['runners'][race_id]
                df_runners = pd.DataFrame(runners)
            except:
                df_runners = pd.DataFrame()

            # WebTips
            try:
                webTips = props['props']['pageProps']['initialState'].get('currentPageState', {}).get('webTips', [])
                df_webTips = pd.DataFrame(webTips).reset_index()
            except:
                df_webTips = pd.DataFrame()

            # Odds
            try:
                odds = get_operator_data_odds(
                    props['props']['pageProps']['initialState'].get('currentPageState', {}).get('betinRaceOdd', {}).get('odds', {})
                )
                df_odds = pd.DataFrame(odds).reset_index()
            except:
                df_odds = pd.DataFrame()

            keys = ['horseId', 'isRunnerState', 'meetingId', 'age', 'hood', 'breederName', 'shoeingFront', 'isEngaged', 'jockeyName', 'draw', 'saddle', 'isSupplemented', 'isRunning', 'weightKg', 'raceDirection', 'numberOfPlaces', 'ownerName', 'raceId', 'uuid', 'raceSpeciality', 'noShoesFirstTime',
                    'raceTotalPrize', 'ranking', 'jockeyUUID', 'totalPrize', 'horseName', 'horseSir', 'trainerUUID', 'meetingName', 'horseUUID', 'ownerUUID', 'trainerName', 'protectionFirstTime', 'raceName', 'jockeyAllowance', 'tongueTie', 'raceIsTQQ', 'sex', 'shoeingBack',
                    'totalWinningPrize','breederId', 'margin', 'jockeyChanged', 'coloursPng', 'shoeing', 'bestImpression', 'comment', 'isPremium', 'blinkers', 'jockeyId', 'raceType', 'ownerId', 'handicapRatingKg', 'horseDam', 'weightChanged', 'claimRating',
                    'trainerId', 'blinkersFirstTime','raceNumber']

            keys2 = ['meetingId', 'raceId', 'text', 'tips']
            keys_odds = ['horseId', 'horseNumber', 'liveOdd', 'referenceOdd', 'isFavorite', 'runnerId', 'liveOddDateTime', 'referenceOddDateTime', 'runnerStatus', 'runnerSlug', 'horseName']

            if not df_runners.empty:
                df_runners = df_runners.reindex(columns=keys, fill_value=np.nan)
                new_runners_dfs.append(df_runners)

            if not df_webTips.empty:
                df_webTips = df_webTips.reindex(columns=keys2, fill_value=np.nan)
                new_webTips_dfs.append(df_webTips)

            if not df_odds.empty:
                df_odds = df_odds.reindex(columns=keys_odds, fill_value=np.nan)
                df_odds['meetingId'] = meeting_id
                df_odds['raceId'] = race_id
                df_odds['timestamp'] = now.strftime('%Y-%m-%d %H:%M:%S')
                new_odds_dfs.append(df_odds)

        # Overwrite runners for updated races
        if new_runners_dfs:
            df_runners_new = pd.concat(new_runners_dfs, ignore_index=True)
            updated_race_ids = races_to_update['id_race'].astype(str).tolist()
            runners_tdy = runners_tdy[~runners_tdy['raceId'].astype(str).isin(updated_race_ids)]
            runners_tdy = pd.concat([runners_tdy, df_runners_new], ignore_index=True)
            runners_tdy.to_parquet("./runners_tdy.parquet", engine='fastparquet', index=None)

        # Overwrite webTips for updated races
        if new_webTips_dfs:
            df_webTips_new = pd.concat(new_webTips_dfs, ignore_index=True)
            updated_race_ids = races_to_update['id_race'].astype(str).tolist()
            webTips_tdy = webTips_tdy[~webTips_tdy['raceId'].astype(str).isin(updated_race_ids)]
            webTips_tdy = pd.concat([webTips_tdy, df_webTips_new], ignore_index=True)
            webTips_tdy.to_parquet("./webTips_tdy.parquet", engine='pyarrow', index=None)

        # Append odds (keep history)
        if new_odds_dfs:
            df_odds_new = pd.concat(new_odds_dfs, ignore_index=True)
            odds_tdy = pd.concat([odds_tdy, df_odds_new], ignore_index=True)
            odds_tdy.to_parquet("./odds_tdy.parquet", engine='pyarrow', index=None)

        print("Update complete!")
        return True

    finally:
        pass  # driver.quit() not needed with Playwright

In [ ]:
if mode in ('get_todays_races', 'get_both'):
  get_today()